# Sequence Timeseries Arch Extension

Ce notebook reprend le script `sequence_timeseries_arch_extension.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Compare des architectures temporelles candidates pour prediction live.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Additional time-series-specific danger sequence architecture tests.
- Commande de reproduction referencee : time-series architecture extension.
- Artefacts controles : Additional time-series architecture extension exists. (`runs/exp_040_timeseries_arch_extension/metrics/timeseries_arch_extension_metrics.csv`).
- Run par defaut : `runs/exp_040_timeseries_arch_extension`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "sequence_timeseries_arch_extension.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score
from torch.utils.data import DataLoader

from ml_pipeline import HORIZONS, ROOT, safe_auc, write_json
from sequence_experiments import (
    FocalBCEWithLogitsLoss,
    SequenceDataset,
    SmoothedBCEWithLogitsLoss,
    append_report,
    evaluate_catalogue_model,
    make_run_dir,
    predict_model,
    set_seed,
)


## Classe `ModelSpec`

Cette cellule definit `ModelSpec`. Elle prepare une partie du script.

In [ ]:
@dataclass(frozen=True)
class ModelSpec:
    name: str
    kind: str
    loss: str
    augment: bool


## Classe `ResidualConvBlock`

Cette cellule definit `ResidualConvBlock`. Elle prepare une partie du script.

In [ ]:
class ResidualConvBlock(nn.Module):
    def __init__(self, channels, kernel_size, dilation=1, dropout=0.15):
        super().__init__()
        padding = dilation * (kernel_size // 2)
        self.net = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size, padding=padding, dilation=dilation),
            nn.BatchNorm1d(channels),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(channels, channels, kernel_size, padding=padding, dilation=dilation),
            nn.BatchNorm1d(channels),
        )
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(x + self.net(x))


## Classe `ResNet1D`

Cette cellule definit `ResNet1D`. Elle prepare une partie du script.

In [ ]:
class ResNet1D(nn.Module):
    def __init__(self, input_dim, out_dim):
        super().__init__()
        self.stem = nn.Conv1d(input_dim, 96, kernel_size=1)
        self.blocks = nn.Sequential(
            ResidualConvBlock(96, 7, 1, 0.12),
            ResidualConvBlock(96, 5, 2, 0.12),
            ResidualConvBlock(96, 3, 4, 0.15),
            ResidualConvBlock(96, 3, 8, 0.15),
        )
        self.head = nn.Sequential(nn.Linear(96 * 3, 128), nn.GELU(), nn.Dropout(0.25), nn.Linear(128, out_dim))

    def forward(self, x):
        z = self.blocks(self.stem(x.transpose(1, 2)))
        pooled = torch.cat([z.mean(dim=-1), z.amax(dim=-1), z[:, :, -1]], dim=1)
        return self.head(pooled)


## Classe `InceptionBlock`

Cette cellule definit `InceptionBlock`. Elle prepare une partie du script.

In [ ]:
class InceptionBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.12):
        super().__init__()
        branch_channels = out_channels // 4
        self.bottleneck = nn.Conv1d(in_channels, branch_channels, kernel_size=1)
        self.conv3 = nn.Conv1d(branch_channels, branch_channels, kernel_size=3, padding=1)
        self.conv7 = nn.Conv1d(branch_channels, branch_channels, kernel_size=7, padding=3)
        self.conv15 = nn.Conv1d(branch_channels, branch_channels, kernel_size=15, padding=7)
        self.pool = nn.Sequential(nn.MaxPool1d(kernel_size=3, stride=1, padding=1), nn.Conv1d(in_channels, branch_channels, kernel_size=1))
        self.norm = nn.BatchNorm1d(branch_channels * 4)
        self.dropout = nn.Dropout(dropout)
        self.shortcut = nn.Conv1d(in_channels, branch_channels * 4, kernel_size=1) if in_channels != branch_channels * 4 else nn.Identity()

    def forward(self, x):
        z = self.bottleneck(x)
        out = torch.cat([self.conv3(z), self.conv7(z), self.conv15(z), self.pool(x)], dim=1)
        return F.gelu(self.shortcut(x) + self.dropout(self.norm(out)))


## Classe `InceptionTimeLite`

Cette cellule definit `InceptionTimeLite`. Elle prepare une partie du script.

In [ ]:
class InceptionTimeLite(nn.Module):
    def __init__(self, input_dim, out_dim):
        super().__init__()
        self.blocks = nn.Sequential(
            InceptionBlock(input_dim, 128, 0.10),
            InceptionBlock(128, 128, 0.12),
            InceptionBlock(128, 128, 0.15),
        )
        self.head = nn.Sequential(nn.Linear(128 * 2, 128), nn.GELU(), nn.Dropout(0.25), nn.Linear(128, out_dim))

    def forward(self, x):
        z = self.blocks(x.transpose(1, 2))
        pooled = torch.cat([z.mean(dim=-1), z.amax(dim=-1)], dim=1)
        return self.head(pooled)


## Classe `SeparableTCNBlock`

Cette cellule definit `SeparableTCNBlock`. Elle prepare une partie du script.

In [ ]:
class SeparableTCNBlock(nn.Module):
    def __init__(self, channels, dilation, dropout=0.16):
        super().__init__()
        self.depthwise = nn.Conv1d(channels, channels, kernel_size=5, padding=2 * dilation, dilation=dilation, groups=channels)
        self.pointwise = nn.Conv1d(channels, channels, kernel_size=1)
        self.norm = nn.BatchNorm1d(channels)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        z = self.depthwise(x)
        z = self.pointwise(z)
        z = self.dropout(F.gelu(self.norm(z)))
        return x + z


## Classe `SeparableTCN`

Cette cellule definit `SeparableTCN`. Elle prepare une partie du script.

In [ ]:
class SeparableTCN(nn.Module):
    def __init__(self, input_dim, out_dim):
        super().__init__()
        self.stem = nn.Conv1d(input_dim, 112, kernel_size=1)
        self.blocks = nn.Sequential(*(SeparableTCNBlock(112, dilation) for dilation in [1, 2, 4, 8, 16]))
        self.head = nn.Sequential(nn.Linear(112 * 3, 128), nn.GELU(), nn.Dropout(0.25), nn.Linear(128, out_dim))

    def forward(self, x):
        z = self.blocks(self.stem(x.transpose(1, 2)))
        pooled = torch.cat([z.mean(dim=-1), z.amax(dim=-1), z[:, :, -1]], dim=1)
        return self.head(pooled)


## Classe `AttentionPooling`

Cette cellule definit `AttentionPooling`. Elle prepare une partie du script.

In [ ]:
class AttentionPooling(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.score = nn.Sequential(nn.Linear(dim, dim // 2), nn.Tanh(), nn.Linear(dim // 2, 1))

    def forward(self, x):
        weights = torch.softmax(self.score(x).squeeze(-1), dim=1)
        return torch.sum(x * weights.unsqueeze(-1), dim=1)


## Classe `BiGRUAttention`

Cette cellule definit `BiGRUAttention`. Elle prepare une partie du script.

In [ ]:
class BiGRUAttention(nn.Module):
    def __init__(self, input_dim, out_dim):
        super().__init__()
        self.rnn = nn.GRU(input_dim, 80, num_layers=2, batch_first=True, dropout=0.20, bidirectional=True)
        self.pool = AttentionPooling(160)
        self.head = nn.Sequential(nn.Linear(160 * 2, 128), nn.GELU(), nn.Dropout(0.25), nn.Linear(128, out_dim))

    def forward(self, x):
        z, _ = self.rnn(x)
        pooled = torch.cat([self.pool(z), z[:, -1]], dim=1)
        return self.head(pooled)


## Classe `ConvTransformerLite`

Cette cellule definit `ConvTransformerLite`. Elle prepare une partie du script.

In [ ]:
class ConvTransformerLite(nn.Module):
    def __init__(self, seq_len, input_dim, out_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(input_dim, 96, kernel_size=5, padding=2),
            nn.BatchNorm1d(96),
            nn.GELU(),
            nn.Dropout(0.10),
        )
        self.pos = nn.Parameter(torch.zeros(1, seq_len, 96))
        layer = nn.TransformerEncoderLayer(d_model=96, nhead=4, dim_feedforward=192, dropout=0.18, batch_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(layer, num_layers=2)
        self.pool = AttentionPooling(96)
        self.head = nn.Sequential(nn.Linear(96 * 2, 128), nn.GELU(), nn.Dropout(0.25), nn.Linear(128, out_dim))

    def forward(self, x):
        z = self.conv(x.transpose(1, 2)).transpose(1, 2)
        z = self.encoder(z + self.pos[:, : z.shape[1]])
        pooled = torch.cat([self.pool(z), z[:, -1]], dim=1)
        return self.head(pooled)


## Classe `PatchTransformerLite`

Cette cellule definit `PatchTransformerLite`. Elle prepare une partie du script.

In [ ]:
class PatchTransformerLite(nn.Module):
    def __init__(self, seq_len, input_dim, out_dim, patch_len=6, stride=3):
        super().__init__()
        self.patch_len = patch_len
        self.stride = stride
        n_patches = 1 + max(0, (seq_len - patch_len) // stride)
        self.proj = nn.Linear(input_dim * patch_len, 128)
        self.pos = nn.Parameter(torch.zeros(1, n_patches, 128))
        layer = nn.TransformerEncoderLayer(d_model=128, nhead=4, dim_feedforward=256, dropout=0.18, batch_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(layer, num_layers=2)
        self.head = nn.Sequential(nn.Linear(128 * 2, 128), nn.GELU(), nn.Dropout(0.25), nn.Linear(128, out_dim))

    def forward(self, x):
        patches = x.unfold(dimension=1, size=self.patch_len, step=self.stride)
        patches = patches.transpose(2, 3).flatten(start_dim=2)
        z = self.proj(patches) + self.pos[:, : patches.shape[1]]
        z = self.encoder(z)
        pooled = torch.cat([z.mean(dim=1), z[:, -1]], dim=1)
        return self.head(pooled)


## Fonction `make_extension_model`

Cette cellule definit `make_extension_model`. Elle prepare une partie du script.

In [ ]:
def make_extension_model(kind, seq_len, input_dim, out_dim):
    if kind == "resnet1d":
        return ResNet1D(input_dim, out_dim)
    if kind == "inceptiontime":
        return InceptionTimeLite(input_dim, out_dim)
    if kind == "separable_tcn":
        return SeparableTCN(input_dim, out_dim)
    if kind == "bigru_attention":
        return BiGRUAttention(input_dim, out_dim)
    if kind == "conv_transformer":
        return ConvTransformerLite(seq_len, input_dim, out_dim)
    if kind == "patch_transformer":
        return PatchTransformerLite(seq_len, input_dim, out_dim)
    raise ValueError(f"Unknown extension kind: {kind}")


## Fonction `make_loss`

Cette cellule definit `make_loss`. Elle prepare une partie du script.

In [ ]:
def make_loss(loss_name, pos_weight, label_smoothing, focal_gamma):
    if loss_name == "bce":
        return nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    if loss_name == "smooth_bce":
        return SmoothedBCEWithLogitsLoss(pos_weight, label_smoothing)
    if loss_name == "focal":
        return FocalBCEWithLogitsLoss(pos_weight, focal_gamma)
    raise ValueError(f"Unknown loss: {loss_name}")


## Fonction `load_sequence_dataset`

Cette cellule definit `load_sequence_dataset`. Elle prepare une partie du script.

In [ ]:
def load_sequence_dataset(sequence_run):
    sequence_run = Path(sequence_run)
    if not sequence_run.is_absolute():
        sequence_run = ROOT / sequence_run
    data = np.load(sequence_run / "features" / "sequence_dataset.npz")
    meta = pd.read_csv(sequence_run / "features" / "sequence_index.csv")
    X = data["X"].astype(np.float32)
    y = data["y"].astype(np.float32)
    return sequence_run, X, y, meta


## Fonction `train_one`

Cette cellule definit `train_one`. Elle prepare une partie du script.

In [ ]:
def train_one(spec, X, y, meta, run_dir, args, device):
    set_seed(args.seed)
    seq_len, input_dim, out_dim = X.shape[1], X.shape[2], y.shape[1]
    train_idx = np.flatnonzero(meta["split"].to_numpy() == "train")
    val_idx = np.flatnonzero(meta["split"].to_numpy() == "val")
    train_ds = SequenceDataset(X, y, train_idx, augment=spec.augment, seed=args.seed)
    val_ds = SequenceDataset(X, y, val_idx, augment=False, seed=args.seed)
    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, num_workers=0)

    model = make_extension_model(spec.kind, seq_len, input_dim, out_dim).to(device)
    positives = y[train_idx].sum(axis=0)
    negatives = len(train_idx) - positives
    pos_weight = torch.tensor(np.clip(negatives / np.maximum(positives, 1.0), 1.0, 20.0), dtype=torch.float32, device=device)
    criterion = make_loss(spec.loss, pos_weight, args.label_smoothing, args.focal_gamma)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

    best = {"score": -1.0, "epoch": 0, "state": None}
    patience_left = args.patience
    history = []
    start = time.perf_counter()
    h1_idx = HORIZONS.index(1.0)
    for epoch in range(1, args.epochs + 1):
        model.train()
        losses = []
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
            optimizer.step()
            losses.append(float(loss.detach().cpu()))

        val_probs, val_targets = predict_model(model, val_loader, device)
        val_ap = safe_auc(average_precision_score, val_targets[:, h1_idx], val_probs[:, h1_idx])
        val_score = float(val_ap or 0.0)
        scheduler.step(val_score)
        history.append(
            {
                "model": spec.name,
                "kind": spec.kind,
                "loss": spec.loss,
                "augment": bool(spec.augment),
                "epoch": epoch,
                "train_loss": float(np.mean(losses)),
                "val_ap_h1": val_score,
                "lr": float(optimizer.param_groups[0]["lr"]),
            }
        )
        if val_score > best["score"] + 1e-5:
            best = {"score": val_score, "epoch": epoch, "state": {k: v.detach().cpu() for k, v in model.state_dict().items()}}
            patience_left = args.patience
        else:
            patience_left -= 1
        if patience_left <= 0:
            break

    if best["state"] is not None:
        model.load_state_dict(best["state"])
    train_time_s = time.perf_counter() - start
    model_path = run_dir / "models" / f"{spec.name}.pt"
    torch.save(
        {
            "model_name": spec.name,
            "kind": spec.kind,
            "loss": spec.loss,
            "augment": bool(spec.augment),
            "state_dict": model.state_dict(),
            "seq_len": int(seq_len),
            "input_dim": int(input_dim),
            "horizons": HORIZONS,
            "best_epoch": int(best["epoch"]),
            "best_val_ap_h1": float(best["score"]),
        },
        model_path,
    )
    return model, history, train_time_s, model_path.stat().st_size


## Fonction `default_specs`

Cette cellule definit `default_specs`. Elle prepare une partie du script.

In [ ]:
def default_specs():
    return [
        ModelSpec("resnet1d_aug_focal", "resnet1d", "focal", True),
        ModelSpec("inceptiontime_aug_focal", "inceptiontime", "focal", True),
        ModelSpec("separable_tcn_aug_focal", "separable_tcn", "focal", True),
        ModelSpec("bigru_attention_aug_focal", "bigru_attention", "focal", True),
        ModelSpec("conv_transformer_aug_focal", "conv_transformer", "focal", True),
        ModelSpec("patch_transformer_aug_focal", "patch_transformer", "focal", True),
        ModelSpec("inceptiontime_noaug_bce", "inceptiontime", "bce", False),
        ModelSpec("resnet1d_aug_smooth", "resnet1d", "smooth_bce", True),
    ]


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    run_dir = make_run_dir(args.run_name)
    sequence_run, X, y, meta = load_sequence_dataset(args.sequence_run)
    set_seed(args.seed)
    device = torch.device("cuda" if args.device == "auto" and torch.cuda.is_available() else args.device)
    specs = default_specs()
    if args.quick:
        specs = specs[:3]

    write_json(
        run_dir / "metrics" / "timeseries_arch_extension_config.json",
        {
            "sequence_run": str(sequence_run),
            "seq_len": int(X.shape[1]),
            "feature_count": int(X.shape[2]),
            "rows": int(len(meta)),
            "split_counts": {str(k): int(v) for k, v in meta["split"].value_counts().sort_index().items()},
            "device": str(device),
            "specs": [spec.__dict__ for spec in specs],
            "augmentation_policy": "train split only through SequenceDataset stochastic augmentations",
            "split_policy": "parent-video split inherited from source sequence_run",
        },
    )

    all_history = []
    all_comparison = []
    all_sweeps = []
    for spec in specs:
        print(f"training {spec.name} on {device}")
        model, history, train_time_s, model_size_bytes = train_one(spec, X, y, meta, run_dir, args, device)
        all_history.extend(history)
        comparison_rows, sweep_rows = evaluate_catalogue_model(
            spec.name,
            model,
            X,
            y,
            meta,
            run_dir,
            device,
            train_time_s,
            model_size_bytes,
            args.batch_size,
            spec.loss,
        )
        all_comparison.extend(comparison_rows)
        all_sweeps.extend(sweep_rows)
        pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "timeseries_arch_extension_training_history.csv", index=False)
        pd.DataFrame(all_comparison).to_csv(run_dir / "metrics" / "timeseries_arch_extension_metrics.csv", index=False)
        if all_sweeps:
            pd.concat(all_sweeps, ignore_index=True).to_csv(run_dir / "metrics" / "timeseries_arch_extension_threshold_sweeps.csv", index=False)

    metrics = pd.DataFrame(all_comparison)
    val = metrics[(metrics["split"] == "val") & (metrics["horizon_s"] == 1.0)].copy()
    test = metrics[(metrics["split"] == "test") & (metrics["horizon_s"] == 1.0)].copy()
    val["selection_score"] = (
        val["average_precision"].fillna(0)
        + 0.5 * val["best_hit_rate"].fillna(0)
        + 0.2 * val["best_window_precision"].fillna(0)
        - 0.03 * val["best_false_alarms_per_min"].fillna(20).clip(upper=20)
    )
    val = val.sort_values("selection_score", ascending=False)
    test = test.sort_values("average_precision", ascending=False)

    lines = ["# Time-Series Architecture Extension", ""]
    lines.append("Additional leakage-safe danger sequence architectures trained on the existing 60-frame sequence dataset.")
    lines.append("")
    lines.append(f"- Source sequence run: `{sequence_run}`")
    lines.append(f"- Rows: `{len(meta)}`")
    lines.append(f"- Sequence length: `{X.shape[1]}` frames")
    lines.append(f"- Feature count: `{X.shape[2]}`")
    lines.append("- Split policy: parent-video split inherited; no random derived-window splitting.")
    lines.append("- Augmentation policy: train-split-only stochastic sequence augmentation when enabled.")
    lines.append("")
    lines.append("## Validation Ranking")
    lines.append("")
    lines.append("| rank | architecture | loss | AP | hit | FA/min | precision | threshold |")
    lines.append("|---:|---|---|---:|---:|---:|---:|---:|")
    for rank, (_, row) in enumerate(val.iterrows(), start=1):
        lines.append(
            f"| {rank} | {row['architecture']} | {row['loss']} | {row['average_precision']:.3f} | "
            f"{row['best_hit_rate']:.3f} | {row['best_false_alarms_per_min']:.3f} | "
            f"{row['best_window_precision']:.3f} | {row['best_threshold_by_hit_fa']:.2f} |"
        )
    lines.append("")
    lines.append("## Test Diagnostics")
    lines.append("")
    lines.append("| rank | architecture | loss | AP | ROC AUC | hit | FA/min | precision | inference ms/window |")
    lines.append("|---:|---|---|---:|---:|---:|---:|---:|---:|")
    for rank, (_, row) in enumerate(test.iterrows(), start=1):
        lines.append(
            f"| {rank} | {row['architecture']} | {row['loss']} | {row['average_precision']:.3f} | "
            f"{row['roc_auc']:.3f} | {row['best_hit_rate']:.3f} | {row['best_false_alarms_per_min']:.3f} | "
            f"{row['best_window_precision']:.3f} | {row['inference_ms_per_window']:.4f} |"
        )
    lines.append("")
    lines.append("## Interpretation")
    lines.append("")
    lines.append("- This extends the catalogue with time-series-specific residual, InceptionTime-style, separable-TCN, attention-RNN, and patch/conv Transformer variants.")
    lines.append("- Selection should use validation ranking; test ranking is diagnostic only.")
    lines.append("- These models are additional evidence against the old tabular/MLP-only framing, not a replacement for repeated-split stability unless they are later repeated across many parent-video splits.")
    summary_path = run_dir / "timeseries_arch_extension_summary.md"
    summary_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    append_report(run_dir, "Time-Series Architecture Extension", f"- Summary: `{summary_path}`\n- Metrics: `{run_dir / 'metrics' / 'timeseries_arch_extension_metrics.csv'}`")
    print(run_dir)


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Additional time-series-specific danger sequence architecture tests.")
    parser.add_argument("--sequence-run", default="runs/exp_008_sequence_len60_catalogue")
    parser.add_argument("--run-name", default="exp_040_timeseries_arch_extension")
    parser.add_argument("--epochs", type=int, default=24)
    parser.add_argument("--patience", type=int, default=5)
    parser.add_argument("--batch-size", type=int, default=128)
    parser.add_argument("--lr", type=float, default=8e-4)
    parser.add_argument("--weight-decay", type=float, default=1.5e-4)
    parser.add_argument("--label-smoothing", type=float, default=0.05)
    parser.add_argument("--focal-gamma", type=float, default=2.0)
    parser.add_argument("--grad-clip", type=float, default=3.0)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--device", default="auto")
    parser.add_argument("--quick", action="store_true")
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_040_timeseries_arch_extension_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["sequence_timeseries_arch_extension.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
